In [11]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [3]:
%pip -q install pandas mlxtend

In [1]:
from google.colab import files

archivos = files.upload()

Saving bread basket.csv to bread basket (1).csv


In [4]:
import pandas as pd

archivo = next(iter(archivos.keys()))
datos = pd.read_csv(archivo)

print("Archivo:", archivo)
print("Cantidad de filas y columnas:", datos.shape)
print("\nNombres de las columnas:")
print(datos.columns.tolist())

print("\nPrimeras 10 filas:")
display(datos.head(10))

Archivo: bread basket (1).csv
Cantidad de filas y columnas: (20507, 5)

Nombres de las columnas:
['Transaction', 'Item', 'date_time', 'period_day', 'weekday_weekend']

Primeras 10 filas:


,Transaction,Item,date_time,period_day,weekday_weekend
0,1,Bread,30-10-2016 09:58,morning,weekend
1,2,Scandinavian,30-10-2016 10:05,morning,weekend
2,2,Scandinavian,30-10-2016 10:05,morning,weekend
3,3,Hot chocolate,30-10-2016 10:07,morning,weekend
4,3,Jam,30-10-2016 10:07,morning,weekend
5,3,Cookies,30-10-2016 10:07,morning,weekend
6,4,Muffin,30-10-2016 10:08,morning,weekend
7,5,Coffee,30-10-2016 10:13,morning,weekend
8,5,Pastry,30-10-2016 10:13,morning,weekend
9,5,Bread,30-10-2016 10:13,morning,weekend


In [5]:
print("Filas originales:", len(datos))
print("Transacciones únicas:", datos["Transaction"].nunique())
print("Productos diferentes:", datos["Item"].nunique())
print("\nValores faltantes:")
print(datos.isnull().sum())

Filas originales: 20507
Transacciones únicas: 9465
Productos diferentes: 94

Valores faltantes:
Transaction        0
Item               0
date_time          0
period_day         0
weekday_weekend    0
dtype: int64


In [6]:
datos = datos.drop_duplicates(subset=["Transaction", "Item"])

print("Filas después de eliminar duplicados:", len(datos))
print("Transacciones únicas:", datos["Transaction"].nunique())
print("Productos diferentes:", datos["Item"].nunique())

Filas después de eliminar duplicados: 18887
Transacciones únicas: 9465
Productos diferentes: 94


In [7]:
datos["Item"] = datos["Item"].astype(str).str.strip()
datos = datos[datos["Item"] != ""]

print("Filas después de limpiar:", len(datos))
print("Transacciones únicas:", datos["Transaction"].nunique())
print("Productos diferentes:", datos["Item"].nunique())

Filas después de limpiar: 18887
Transacciones únicas: 9465
Productos diferentes: 94


In [8]:
matriz = pd.crosstab(datos["Transaction"], datos["Item"]).clip(upper=1)

print("Dimensiones de la matriz:", matriz.shape)
display(matriz.head())

Dimensiones de la matriz: (9465, 94)


Item,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
from mlxtend.frequent_patterns import apriori

itemsets = apriori(
    matriz.astype(bool),
    min_support=0.01,
    use_colnames=True
)

print("Conjuntos de productos frecuentes:", len(itemsets))
display(itemsets.head(10))

Conjuntos de productos frecuentes: 61


,support,itemsets
0,0.036344,(Alfajores)
1,0.016059,(Baguette)
2,0.327205,(Bread)
3,0.040042,(Brownie)
4,0.103856,(Cake)
5,0.012995,(Chicken Stew)
6,0.478394,(Coffee)
7,0.019440,(Coke)
8,0.054411,(Cookies)
9,0.039197,(Farm House)


In [13]:
itemsets_2 = apriori(
    matriz.astype(bool),
    min_support=0.005,
    use_colnames=True
)

print("Conjuntos frecuentes con soporte 0.005:", len(itemsets_2))

Conjuntos frecuentes con soporte 0.005: 114


In [14]:
itemsets_3 = apriori(
    matriz.astype(bool),
    min_support=0.003,
    use_colnames=True
)

print("Conjuntos frecuentes con soporte 0.003:", len(itemsets_3))

Conjuntos frecuentes con soporte 0.003: 177


In [15]:
from mlxtend.frequent_patterns import association_rules

reglas = association_rules(
    itemsets_3,
    metric="confidence",
    min_threshold=0.3
)

print("Cantidad de reglas generadas:", len(reglas))

Cantidad de reglas generadas: 63


In [16]:
reglas_ordenadas = reglas.sort_values(
    by="lift",
    ascending=False
)

display(
    reglas_ordenadas[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

,antecedents,consequents,support,confidence,lift
11,(Extra Salami or Feta),(Coffee),0.003275,0.815789,1.705267
17,(Keeping It Local),(Coffee),0.005388,0.809524,1.692169
32,(Toast),(Coffee),0.023666,0.704403,1.472431
49,"(Cake, Sandwich)",(Coffee),0.004649,0.676923,1.414990
55,"(Pastry, Hot chocolate)",(Coffee),0.003803,0.666667,1.393551
59,"(Soup, Sandwich)",(Coffee),0.003592,0.653846,1.366752
54,"(Medialuna, Hot chocolate)",(Coffee),0.003064,0.644444,1.347100
28,(Tartine),(Coffee),0.003064,0.630435,1.317815
22,(Salad),(Coffee),0.006550,0.626263,1.309094
51,"(Cookies, Hot chocolate)",(Coffee),0.003698,0.614035,1.283534


In [17]:
cinco_reglas = reglas_ordenadas[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(5).copy()

cinco_reglas["antecedents"] = cinco_reglas["antecedents"].apply(
    lambda x: ", ".join(sorted(x))
)
cinco_reglas["consequents"] = cinco_reglas["consequents"].apply(
    lambda x: ", ".join(sorted(x))
)

display(cinco_reglas.round(4))

,antecedents,consequents,support,confidence,lift
11,Extra Salami or Feta,Coffee,0.0033,0.8158,1.7053
17,Keeping It Local,Coffee,0.0054,0.8095,1.6922
32,Toast,Coffee,0.0237,0.7044,1.4724
49,"Cake, Sandwich",Coffee,0.0046,0.6769,1.4150
55,"Hot chocolate, Pastry",Coffee,0.0038,0.6667,1.3936


1. Extra Salami or Feta → Coffee: la confianza es 81,58 % y el lift es 1,7053. Existe una asociación positiva con la compra de café.

2. Keeping It Local → Coffee: la confianza es 80,95 % y el lift es 1,6922. Quienes compran este producto muestran una mayor tendencia a comprar café.

3. Toast → Coffee: la confianza es 70,44 % y el lift es 1,4724. El café aparece asociado a las compras de tostadas.

4. Cake y Sandwich → Coffee: la confianza es 67,69 % y el lift es 1,4150. Cuando ambos productos están presentes, se observa una asociación positiva con el café.

5. Hot chocolate y Pastry → Coffee: la confianza es 66,67 % y el lift es 1,3936. La combinación se relaciona positivamente con la compra de café.

Conclusión: las cinco reglas tienen un lift superior a 1, lo que indica una asociación positiva. Esto permite proponer promociones combinadas con café, aunque no demuestra causalidad.

In [18]:
reglas_bajas = reglas.sort_values(by="lift", ascending=True)

display(
    reglas_bajas[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(5)
)

,antecedents,consequents,support,confidence,lift
9,(Coke),(Coffee),0.006445,0.331522,0.692989
29,(Tea),(Coffee),0.049868,0.349630,0.730840
37,"(Bread, Brownie)",(Coffee),0.003803,0.352941,0.737762
39,"(Bread, Cookies)",(Coffee),0.005283,0.364964,0.762893
33,(Truffles),(Coffee),0.007501,0.369792,0.772985


Regla rechazada: Coke → Coffee
- Soporte: 0,006445 (0,6445 %).
- Confianza: 33,15 %.
- Lift: 0,6930.

Justificación: Se rechaza esta regla como recomendación comercial porque su lift es inferior a 1. Esto indica que quienes compran Coke presentan una menor probabilidad de comprar Coffee que la probabilidad general de compra de café. Por lo tanto, no existe una asociación positiva que justifique recomendar esta combinación.

In [19]:
from mlxtend.frequent_patterns import association_rules

configuraciones = [
    (0.010, 0.30, itemsets),
    (0.005, 0.30, itemsets_2),
    (0.003, 0.30, itemsets_3)
]

resultados = []

for soporte, confianza, conjuntos in configuraciones:
    reglas_temp = association_rules(
        conjuntos,
        metric="confidence",
        min_threshold=confianza
    )

    resultados.append({
        "Soporte mínimo": soporte,
        "Confianza mínima": confianza,
        "Conjuntos frecuentes": len(conjuntos),
        "Reglas generadas": len(reglas_temp)
    })

comparacion = pd.DataFrame(resultados)
display(comparacion)

,Soporte mínimo,Confianza mínima,Conjuntos frecuentes,Reglas generadas
0,0.010,0.3,61,19
1,0.005,0.3,114,35
2,0.003,0.3,177,63


Análisis comparativo: Al disminuir el soporte mínimo de 0,010 a 0,003, los conjuntos frecuentes aumentaron de 61 a 177 y las reglas generadas aumentaron de 19 a 63. Esto demuestra que un umbral de soporte menor permite identificar más asociaciones, aunque algunas pueden aparecer en pocas transacciones. Por ello, es importante considerar conjuntamente el soporte, la confianza y el lift para seleccionar reglas útiles para la toma de decisiones comerciales.

Se propone evaluar promociones que combinen café con tostadas, pastelería y sándwiches, debido a las asociaciones positivas identificadas mediante Apriori. La regla Toast → Coffee presenta una confianza de 70,44 % y un lift de 1,4724. Se recomienda probar una promoción de tostadas y café y comparar sus resultados de ventas antes de implementarla de forma permanente.

La implementación del algoritmo Apriori permitió identificar asociaciones entre productos del conjunto de datos Bread Basket. Al reducir el soporte mínimo, aumentó la cantidad de conjuntos frecuentes y reglas generadas. Las métricas de soporte, confianza y lift permitieron distinguir asociaciones positivas de otras menos útiles. Los resultados pueden orientar promociones comerciales, aunque deben validarse con datos reales de ventas antes de tomar decisiones definitivas.

In [20]:
comparacion.to_csv("comparacion_configuraciones.csv", index=False)
cinco_reglas.to_csv("cinco_reglas_principales.csv", index=False)

regla_rechazada = reglas_bajas.head(1)
regla_rechazada.to_csv("regla_rechazada.csv", index=False)

print("Archivos guardados correctamente:")
print("- comparacion_configuraciones.csv")
print("- cinco_reglas_principales.csv")
print("- regla_rechazada.csv")

Archivos guardados correctamente:
- comparacion_configuraciones.csv
- cinco_reglas_principales.csv
- regla_rechazada.csv


In [21]:
from google.colab import files
files.download("comparacion_configuraciones.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from google.colab import files
files.download("cinco_reglas_principales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files
files.download("regla_rechazada.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>